# Ch. 4 · TP2 Réseaux convolutifs avec PyTorch

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch04-tp2-pytorch-cnn.ipynb)

**IA pour les sciences — M1.** Écrire une boucle d’entraînement, comparer MLP et CNN sur des images, surveiller la validation.

**Durée indicative :** 1 h 30 (partie B du TP de 3 h) — CPU suffisant, GPU facultatif

**Objectifs**

- Manipuler des tenseurs et un `DataLoader` PyTorch.
- Écrire soi-même la boucle entraînement / validation (perte, rétropropagation, optimiseur).
- Comparer un MLP et un CNN à nombre de paramètres comparable.
- Appliquer l’early stopping et évaluer une seule fois sur le test.
- Visualiser filtres et erreurs pour interpréter le modèle.

**Prérequis :** TP1 du chapitre 4. Sur Colab, PyTorch est préinstallé ; en local : `pip install torch`.

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 0. Installation et reproductibilité

Les images sont les **chiffres manuscrits 8×8** de scikit-learn (1 797 images, aucun téléchargement nécessaire).
Le même code s’applique à des images de microscopie, de lames minces ou de télédétection, avec des images plus grandes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '— calcul sur', device)

## 1. Données : images → tenseurs

Un lot d’images a la forme `(N, C, H, W)` : nombre d’images, canaux (1 en niveaux de gris), hauteur, largeur.
On sépare **entraînement / validation / test** (60 / 20 / 20). La normalisation est calculée sur l’entraînement seulement.

In [ ]:
digits = load_digits()
images = digits.images.astype(np.float32) / 16.0       # pixels entre 0 et 1
y = digits.target
X_tmp, X_test, y_tmp, y_test = train_test_split(images, y, test_size=0.2, stratify=y, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=0)
moy, et = X_train.mean(), X_train.std()

def vers_tenseurs(Xa, ya):
    return TensorDataset(torch.tensor((Xa - moy) / et).unsqueeze(1), torch.tensor(ya, dtype=torch.long))

train_ds, val_ds, test_ds = vers_tenseurs(X_train, y_train), vers_tenseurs(X_val, y_val), vers_tenseurs(X_test, y_test)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl, test_dl = DataLoader(val_ds, batch_size=256), DataLoader(test_ds, batch_size=256)
print('Entraînement', len(train_ds), '| validation', len(val_ds), '| test', len(test_ds))
print('Forme d’un lot :', next(iter(train_dl))[0].shape)

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, img, lab in zip(axes, X_train, y_train):
    ax.imshow(img, cmap='gray_r'); ax.set_title(lab); ax.axis('off')
plt.show()

## 2. Deux architectures

- **MLP** : l’image est aplatie en 64 nombres ; la structure spatiale est perdue.
- **CNN** : des **filtres 3×3** partagés glissent sur l’image (convolution), détectant des motifs locaux (traits, courbes)
  où qu’ils soient. Le *max pooling* réduit la résolution et apporte une petite invariance aux translations.

Taille de sortie d’une convolution : $H_{out} = H_{in} - k + 2p + 1$ (pas de 1, noyau $k$, remplissage $p$).

In [ ]:
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 10),
)
cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),   # 16 x 8 x 8
    nn.MaxPool2d(2),                                          # 16 x 4 x 4
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),  # 32 x 4 x 4
    nn.MaxPool2d(2),                                          # 32 x 2 x 2
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(32 * 2 * 2, 10),
)

def n_parametres(modele):
    return sum(p.numel() for p in modele.parameters())

print('Paramètres MLP :', n_parametres(mlp))
print('Paramètres CNN :', n_parametres(cnn))

## 3. La boucle d’entraînement, ligne par ligne

Pour chaque lot : (1) **propagation avant** ; (2) calcul de la **perte** (entropie croisée, qui inclut la softmax) ;
(3) `zero_grad()` puis `backward()` : **rétropropagation** des gradients ; (4) `step()` : mise à jour par l’optimiseur **Adam**.
On évalue la validation à chaque époque, en mode `eval()` (dropout désactivé) et sans calcul de gradient.
On conserve les poids de la **meilleure époque de validation** (early stopping).

In [ ]:
import copy

def evaluer(modele, loader, critere):
    modele.eval()
    perte, corrects, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            sortie = modele(xb)
            perte += critere(sortie, yb).item() * len(yb)
            corrects += (sortie.argmax(1) == yb).sum().item()
            n += len(yb)
    return perte / n, corrects / n

def entrainer(modele, epoques=40, lr=1e-3, patience=8):
    modele.to(device)
    critere = nn.CrossEntropyLoss()
    optimiseur = torch.optim.Adam(modele.parameters(), lr=lr)
    hist = {'train': [], 'val': [], 'val_acc': []}
    meilleur, meilleurs_poids, attente = np.inf, None, 0
    for epoque in range(epoques):
        modele.train()
        total, n = 0.0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimiseur.zero_grad()
            perte = critere(modele(xb), yb)
            perte.backward()
            optimiseur.step()
            total += perte.item() * len(yb); n += len(yb)
        perte_val, acc_val = evaluer(modele, val_dl, critere)
        hist['train'].append(total / n); hist['val'].append(perte_val); hist['val_acc'].append(acc_val)
        if perte_val < meilleur:
            meilleur, meilleurs_poids, attente = perte_val, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= patience:
                break
    modele.load_state_dict(meilleurs_poids)
    return hist

historiques = {}
for nom, modele in [('MLP', mlp), ('CNN', cnn)]:
    historiques[nom] = entrainer(modele)
    h = historiques[nom]
    print(f'{nom} : {len(h["train"])} époques, meilleure exactitude de validation {max(h["val_acc"]):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for nom, h in historiques.items():
    axes[0].plot(h['train'], label=f'{nom} entraînement')
    axes[0].plot(h['val'], '--', label=f'{nom} validation')
    axes[1].plot(h['val_acc'], label=nom)
axes[0].set(xlabel='Époque', ylabel='Entropie croisée', title='Courbes de perte'); axes[0].legend()
axes[1].set(xlabel='Époque', ylabel='Exactitude', title='Exactitude de validation'); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Évaluation finale — une seule fois sur le test

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

critere = nn.CrossEntropyLoss()
for nom, modele in [('MLP', mlp), ('CNN', cnn)]:
    _, acc = evaluer(modele, test_dl, critere)
    print(f'{nom} — exactitude test : {acc:.3f}')

cnn.eval()
with torch.no_grad():
    xt = test_ds.tensors[0].to(device)
    pred = cnn(xt).argmax(1).cpu().numpy()
ConfusionMatrixDisplay(confusion_matrix(y_test, pred)).plot(cmap='Blues')
plt.title('CNN — matrice de confusion (test)'); plt.show()

## 5. Interpréter : filtres appris et erreurs

Les filtres de la première couche sont de petits détecteurs de motifs. Les images mal classées révèlent souvent
des cas réellement ambigus (ou des erreurs d’étiquetage) : les examiner fait partie de l’analyse scientifique.

In [ ]:
filtres = cnn[0].weight.detach().cpu().numpy()[:, 0]
fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, f in zip(axes.ravel(), filtres):
    ax.imshow(f, cmap='RdBu_r'); ax.axis('off')
plt.suptitle('Filtres 3×3 de la première couche'); plt.show()

erreurs = np.where(pred != y_test)[0]
print(len(erreurs), 'erreurs sur', len(y_test))
fig, axes = plt.subplots(1, min(8, len(erreurs)), figsize=(12, 1.8), squeeze=False)
for ax, i in zip(axes[0], erreurs[:8]):
    ax.imshow(X_test[i], cmap='gray_r'); ax.set_title(f'vrai {y_test[i]} / prédit {pred[i]}', fontsize=8); ax.axis('off')
plt.show()

### ✏️ Exercice 1 — Robustesse à une translation

Décalez toutes les images de test d’**un pixel vers la droite** (`np.roll(X_test, 1, axis=2)`), évaluez MLP et CNN. Lequel résiste le mieux ? Pourquoi ?

In [ ]:
X_decale = np.roll(X_test, 1, axis=2)
decale_ds = vers_tenseurs(X_decale, y_test)
for nom, modele in [('MLP', mlp), ('CNN', cnn)]:
    _, acc = evaluer(modele, DataLoader(decale_ds, batch_size=256), critere)
    print(nom, 'exactitude (images décalées) :', round(acc, 3))

<details><summary><b>Voir le corrigé</b></summary>

```python
X_decale = np.roll(X_test, 1, axis=2)
decale_ds = vers_tenseurs(X_decale, y_test)
for nom, modele in [('MLP', mlp), ('CNN', cnn)]:
    _, acc = evaluer(modele, DataLoader(decale_ds, batch_size=256), critere)
    print(nom, 'exactitude (images décalées) :', round(acc, 3))
```

Le CNN, avec ses filtres partagés et son pooling, perd généralement moins que le MLP, dont chaque poids est attaché à une position de pixel. C’est un **décalage de distribution** élémentaire : en sciences, un changement de microscope ou de cadrage produit le même type d’effet. Cette cellule est déjà complète : exécutez-la et interprétez.

</details>

### ✏️ Exercice 2 — Concevoir son CNN

Construisez un CNN avec **8** filtres puis **16** filtres (au lieu de 16 et 32). Comptez ses paramètres, entraînez-le avec `entrainer` et comparez son exactitude de validation. Le gain de capacité vaut-il le surcoût ?

In [ ]:
petit_cnn = nn.Sequential(
    nn.Conv2d(1, ..., kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(..., ..., kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(... * 2 * 2, 10),
)
print('Paramètres :', n_parametres(petit_cnn))
h = entrainer(petit_cnn)
print('Meilleure exactitude de validation :', round(max(h['val_acc']), 3))

<details><summary><b>Voir le corrigé</b></summary>

```python
petit_cnn = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(16 * 2 * 2, 10),
)
print('Paramètres :', n_parametres(petit_cnn))
h = entrainer(petit_cnn)
print('Meilleure exactitude de validation :', round(max(h['val_acc']), 3))
```

Avec environ quatre fois moins de paramètres, l’exactitude reste souvent proche. On compare sur la **validation**, jamais sur le test.

</details>

## Pour aller plus loin : transfert d’apprentissage

Pour des images scientifiques réelles et peu nombreuses, on part souvent d’un réseau pré-entraîné (par exemple
`torchvision.models.resnet18(weights='DEFAULT')`), dont on remplace la dernière couche avant de l’ajuster.
Il faut alors **vérifier l’écart de domaine** : des photos du quotidien ne ressemblent pas à des lames minces.

## Interprétation scientifique (à rédiger)

1. Comparez MLP et CNN : exactitude test, nombre de paramètres, nombre d’époques.
2. Que montrent les courbes de validation sur le moment où l’apprentissage devrait s’arrêter ?
3. Les erreurs du CNN sont-elles compréhensibles pour un humain ?
4. Qu’indique l’expérience de translation pour une application à des images de laboratoire réelles ?